# pyda Repository — Submit, Retrieve, and Search

This notebook demonstrates how to connect to an LTPDA repository, submit data objects,
retrieve them by ID or collection, and search the repository.

pyda connects **directly to MySQL** — the same way the MATLAB toolbox does via JDBC.
No REST API is involved.

**Before running this notebook**, make sure you have a running LTPDA repository database
accessible from this machine (directly or via an SSH tunnel you have already set up).

---

⚠️ **Do not save this notebook with real credentials filled in.** Use `os.environ` or
a `.env` file (see the README) so credentials are never stored in the notebook file.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from pyda.tsdata import TSData
from pyda.fsdata import FSData
from pyda.dsp.spectral import psd
from pyda.repo import LTPDARepository

## 1. Connecting to the repository

Three ways to connect — pick whichever suits your setup.

In [ ]:
# Option A: explicit credentials (simplest for one-off notebooks)
# repo = LTPDARepository('db.host.com', 'my_repo', 'alice', 'mysql_secret')

# Option B: SSH tunnel already running on local port 3307
#   ssh -L 3307:db.internal:3306 gateway.host -N &
# repo = LTPDARepository('localhost', 'my_repo', 'alice', 'mysql_secret', port=3307)

# Option C: environment variables (recommended — keeps credentials out of the notebook)
#   Set LTPDA_HOST, LTPDA_DB, LTPDA_USER, LTPDA_PASS (and optionally LTPDA_PORT)
#   in your shell, .env file, or CI secret store.
import os
os.environ.setdefault('LTPDA_HOST', 'localhost')
os.environ.setdefault('LTPDA_PORT', '3306')
os.environ.setdefault('LTPDA_DB',   'ltpda_repo')
os.environ.setdefault('LTPDA_USER', 'alice')
os.environ.setdefault('LTPDA_PASS', 'mysql_secret')

repo = LTPDARepository.from_env()
repo.connect()
print('Connected.')

## 2. List accessible databases

Useful for discovering which repository schemas are accessible with the current credentials.

In [ ]:
dbs = repo.list_databases()
print('Accessible databases:', dbs)

## 3. Submit a TSData object

Create a 1-hour segment of white noise at 10 Hz, give it an absolute start time, and submit.

In [ ]:
ts = TSData.randn(nsecs=3600, fs=10, name='ACC_X', yunits='m/s^2')
ts.t0 = datetime(2024, 1, 15, 0, 0, 0)   # UTC absolute start time
ts.description = 'Simulated accelerometer X-axis noise'

print(ts)

In [ ]:
result = repo.submit(
    ts,
    experiment_title='Noise floor run Jan 15',
    experiment_desc='Accelerometer noise at rest on optical bench — X axis',
    analysis_desc='No processing applied; raw data submission for baseline',
    quantity='acceleration',
    keywords='noise, accelerometer, baseline',
)

print(f'Submitted: id={result.id}, uuid={result.uuid}')
ts_id = result.id

## 4. Submit an FSData object (PSD)

Compute the power spectral density of the noise and store it in the same experiment.

In [ ]:
Pxx = psd(ts, navs=50, window='BH92')
Pxx.name = 'ACC_X_PSD'

psd_result = repo.submit(
    Pxx,
    experiment_title='Noise floor run Jan 15',
    experiment_desc='Accelerometer noise at rest on optical bench — X axis',
    analysis_desc='Welch PSD, 50 averages, BH92 window, no detrending',
    quantity='acceleration PSD',
    keywords='psd, accelerometer, BH92',
)

print(f'PSD submitted: id={psd_result.id}')
psd_id = psd_result.id

## 5. Submit multiple objects as a collection

Submitting several objects at once wraps them in a MySQL transaction and creates a collection
automatically. All returned `SubmitResult` objects share the same `cid`.

In [ ]:
ts_y = TSData.randn(nsecs=3600, fs=10, name='ACC_Y', yunits='m/s^2')
ts_y.t0 = datetime(2024, 1, 15, 0, 0, 0)

ts_z = TSData.randn(nsecs=3600, fs=10, name='ACC_Z', yunits='m/s^2')
ts_z.t0 = datetime(2024, 1, 15, 0, 0, 0)

results = repo.submit(
    ts, ts_y, ts_z,
    experiment_title='Three-axis noise floor Jan 15',
    experiment_desc='Simultaneous X Y Z accelerometer noise at rest on optical bench',
    analysis_desc='No processing; raw data submission for three-axis baseline',
    quantity='acceleration',
    keywords='noise, accelerometer, 3-axis',
)

collection_id = results[0].cid
ids_in_collection = [r.id for r in results]
print(f'Collection id: {collection_id}')
print(f'Object ids:    {ids_in_collection}')

## 6. Search the repository

`repo.find()` returns a list of `SearchResult` objects ordered by submission date, newest first.

In [ ]:
# All accelerometer objects
hits = repo.find(name='ACC%')
print(f'Found {len(hits)} objects matching ACC%')
for h in hits[:10]:
    print(f'  id={h.id:5d}  name={h.name!r:15s}  submitted={h.submitted}  t_start={h.t_start}')

In [ ]:
# Filter by author and date
hits2 = repo.find(
    name='ACC%',
    date_from='2024-01-01',
    date_to='2024-02-01',
)
print(f'Found {len(hits2)} objects in January 2024')

## 7. Retrieve by ID

In [ ]:
ts_back = repo.retrieve(ts_id)
print(ts_back)

In [ ]:
# Retrieve multiple IDs at once
psd_back = repo.retrieve(psd_id)
psd_back.sqrt().loglog()
plt.title('Retrieved PSD (ASD)')
plt.show()

## 8. Retrieve a collection

In [ ]:
axes_back = repo.retrieve(cid=collection_id)
print(f'Retrieved {len(axes_back)} objects from collection {collection_id}')
for obj in axes_back:
    print(f'  {obj.name}')

## 9. Full round-trip check

Verify that the data retrieved from the repository is numerically identical to what was submitted.

In [ ]:
ts_rt = repo.retrieve(ts_id)

data_match = np.allclose(ts.yaxis.data, ts_rt.yaxis.data)
fs_match   = ts.fs() == ts_rt.fs()
t0_match   = ts.t0 == ts_rt.t0

print(f'yaxis.data identical: {data_match}')
print(f'fs identical:         {fs_match}')
print(f't0 identical:         {t0_match}')

assert data_match, 'Round-trip data mismatch!'
assert fs_match,   'Round-trip fs mismatch!'
assert t0_match,   'Round-trip t0 mismatch!'
print('Round-trip check passed.')

## 10. Metadata retrieval (no binary download)

In [ ]:
metas = repo.get_metadata(ts_id, psd_id)
for m in metas:
    print(f'id={m.id}  name={m.name!r}  type={m.data_type}  fs={m.fs}  nsecs={m.nsecs}  t0={m.t0}')

## 11. Disconnect

In [ ]:
repo.close()
print('Connection closed.')